# 🎯 Daily Churn Probability Scoring Pipeline
**Purpose:** Move from EDA correlations to **actionable triggers**.

This notebook trains an XGBoost + Random Forest classifier on engineered features (RFM, PlayIntensity, ProgressionRate) and outputs a **scored player roster** where every active player gets a Churn Probability (0–100%). Players above 75% are exported to a CRM-ready CSV for automated win-back campaigns.

### Pipeline Steps
1. Feature engineering (RFM + behavioral)
2. Train/test split
3. Train XGBoost & Random Forest
4. Evaluate (AUC-ROC, Precision, Recall, F1)
5. Score entire player base
6. Export >75% churn-risk list → CRM CSV
7. Feature importance for explainability

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (classification_report, roc_auc_score, roc_curve,
                             precision_recall_curve, confusion_matrix, f1_score)
from sklearn.preprocessing import LabelEncoder
import joblib, os, warnings
warnings.filterwarnings('ignore')

%matplotlib inline
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

spark = SparkSession.builder.appName('ChurnScoringPipeline') \
    .master('local[*]').config('spark.driver.memory','4g').getOrCreate()
spark.sparkContext.setLogLevel('ERROR')
print(f'Spark {spark.version} ready.')

---
## 1. Feature Engineering Pipeline

In [ ]:
df = spark.read.csv('../data/raw/online_gaming_behavior_dataset.csv', header=True, inferSchema=True)

# Target
df = df.withColumn('Churn_Risk', F.when(F.col('EngagementLevel')=='Low',1).otherwise(0))

# Engineered features
df = df.withColumn('TotalWeeklyMinutes', F.col('SessionsPerWeek') * F.col('AvgSessionDurationMinutes'))
df = df.withColumn('PlayIntensity', F.col('PlayTimeHours') * F.col('SessionsPerWeek'))
df = df.withColumn('ProgressionRate', F.col('AchievementsUnlocked') / (F.col('PlayerLevel') + 1))
df = df.withColumn('MonetaryProxy', F.col('PlayTimeHours') * (F.col('InGamePurchases') + 1))

# RFM scores
r_q = df.approxQuantile('AvgSessionDurationMinutes', [0.25,0.5,0.75], 0.01)
f_q = df.approxQuantile('SessionsPerWeek', [0.25,0.5,0.75], 0.01)
m_q = df.approxQuantile('MonetaryProxy', [0.25,0.5,0.75], 0.01)

def qscore(col, q):
    return F.when(F.col(col)<=q[0],1).when(F.col(col)<=q[1],2).when(F.col(col)<=q[2],3).otherwise(4)

df = df.withColumn('R_Score', qscore('AvgSessionDurationMinutes', r_q))
df = df.withColumn('F_Score', qscore('SessionsPerWeek', f_q))
df = df.withColumn('M_Score', qscore('MonetaryProxy', m_q))
df = df.withColumn('RFM_Total', F.col('R_Score') + F.col('F_Score') + F.col('M_Score'))

# Player Persona via K-Means
seg_cols = ['R_Score','F_Score','M_Score','PlayTimeHours','SessionsPerWeek','PlayerLevel','AchievementsUnlocked']
assembler = VectorAssembler(inputCols=seg_cols, outputCol='feat_raw', handleInvalid='skip')
scaler = StandardScaler(inputCol='feat_raw', outputCol='feat_sc', withStd=True, withMean=True)
df_v = assembler.transform(df)
df_s = scaler.fit(df_v).transform(df_v)
km = KMeans(k=4, seed=42, featuresCol='feat_sc', predictionCol='Segment')
df = km.fit(df_s).transform(df_s)

cp = df.groupBy('Segment').agg(F.mean('RFM_Total').alias('AvgRFM')).orderBy('AvgRFM').toPandas()
names = ['At-Risk Newbies','Weekend Warriors','Grinders','Whales']
seg_map = dict(zip(cp['Segment'].values, names))
expr = F.lit('Unknown')
for sid, nm in seg_map.items():
    expr = F.when(F.col('Segment')==int(sid), nm).otherwise(expr)
df = df.withColumn('Player_Persona', expr)

print('Feature engineering complete.')
df.select('PlayerID','Churn_Risk','PlayIntensity','ProgressionRate','RFM_Total','Player_Persona').show(5)

---
## 2. Prepare ML Dataset
Convert to pandas for sklearn/xgboost training. Encode categoricals.

In [ ]:
feature_cols = ['Age','PlayTimeHours','SessionsPerWeek','AvgSessionDurationMinutes',
                'PlayerLevel','AchievementsUnlocked','InGamePurchases',
                'TotalWeeklyMinutes','PlayIntensity','ProgressionRate',
                'MonetaryProxy','R_Score','F_Score','M_Score','RFM_Total']
cat_cols = ['Gender','Location','GameGenre','GameDifficulty','Player_Persona']
id_col = 'PlayerID'
target = 'Churn_Risk'

pdf = df.select([id_col, target] + feature_cols + cat_cols).toPandas()

# Encode categoricals
le_dict = {}
for c in cat_cols:
    le = LabelEncoder()
    pdf[c + '_enc'] = le.fit_transform(pdf[c].astype(str))
    le_dict[c] = le

enc_cols = [c + '_enc' for c in cat_cols]
all_features = feature_cols + enc_cols

X = pdf[all_features].values
y = pdf[target].values
player_ids = pdf[id_col].values

X_train, X_test, y_train, y_test, ids_train, ids_test = train_test_split(
    X, y, player_ids, test_size=0.2, random_state=42, stratify=y)

print(f'Training set: {X_train.shape[0]:,} | Test set: {X_test.shape[0]:,}')
print(f'Churn rate (train): {y_train.mean()*100:.1f}% | Churn rate (test): {y_test.mean()*100:.1f}%')
print(f'Features: {len(all_features)}')

---
## 3. Train Models: XGBoost & Random Forest

In [ ]:
# XGBoost
xgb = XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=(y_train==0).sum()/(y_train==1).sum(),
    eval_metric='auc', random_state=42, use_label_encoder=False
)
xgb.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=0)

# Random Forest
rf = RandomForestClassifier(
    n_estimators=300, max_depth=12, class_weight='balanced',
    random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)

print('Both models trained successfully.')

---
## 4. Model Evaluation

In [ ]:
models = {'XGBoost': xgb, 'Random Forest': rf}
results = {}

for name, model in models.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_prob)
    f1 = f1_score(y_test, y_pred)
    results[name] = {'y_pred': y_pred, 'y_prob': y_prob, 'auc': auc, 'f1': f1}
    print(f'\n=== {name} ===')
    print(f'AUC-ROC: {auc:.4f} | F1-Score: {f1:.4f}')
    print(classification_report(y_test, y_pred, target_names=['Retained', 'Churned']))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 6))
colors = {'XGBoost': '#e74c3c', 'Random Forest': '#3498db'}

# ROC Curves
for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    axes[0].plot(fpr, tpr, label=f"{name} (AUC={res['auc']:.3f})", color=colors[name], linewidth=2)
axes[0].plot([0,1],[0,1],'k--',alpha=0.5)
axes[0].set_xlabel('False Positive Rate'); axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves', fontweight='bold'); axes[0].legend()

# Precision-Recall Curves
for name, res in results.items():
    prec, rec, _ = precision_recall_curve(y_test, res['y_prob'])
    axes[1].plot(rec, prec, label=name, color=colors[name], linewidth=2)
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curves', fontweight='bold'); axes[1].legend()

# Confusion Matrix (best model)
best = max(results, key=lambda k: results[k]['auc'])
cm = confusion_matrix(y_test, results[best]['y_pred'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[2],
            xticklabels=['Retained','Churned'], yticklabels=['Retained','Churned'])
axes[2].set_xlabel('Predicted'); axes[2].set_ylabel('Actual')
axes[2].set_title(f'Confusion Matrix ({best})', fontweight='bold')

plt.suptitle('Model Performance Comparison', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../data/processed/chart_model_performance.png', bbox_inches='tight')
plt.show()
print(f'\nBest model: {best} (AUC={results[best]["auc"]:.4f})')

---
## 5. Feature Importance: What Drives Churn Predictions?

In [ ]:
best_model = models[best]
importances = best_model.feature_importances_
feat_imp = pd.DataFrame({'Feature': all_features, 'Importance': importances}) \
    .sort_values('Importance', ascending=True).tail(15)

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(feat_imp['Feature'], feat_imp['Importance'], color='#3498db', edgecolor='black')
ax.set_xlabel('Feature Importance')
ax.set_title(f'Top 15 Churn Predictors ({best})', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/chart_feature_importance.png', bbox_inches='tight')
plt.show()

---
## 6. 🎯 Score Entire Player Base (Daily Pipeline Output)
Every player gets a **Churn Probability Score (0–100%)**.

In [ ]:
# Score ALL players using the best model
all_probs = best_model.predict_proba(X)[:, 1]

scored = pdf[[id_col] + cat_cols + feature_cols + [target]].copy()
scored['Churn_Probability_%'] = (all_probs * 100).round(2)
scored['Risk_Tier'] = pd.cut(scored['Churn_Probability_%'],
    bins=[0, 25, 50, 75, 100],
    labels=['Low (0-25%)', 'Medium (25-50%)', 'High (50-75%)', 'Critical (75-100%)'],
    include_lowest=True)

print(f'=== Scored {len(scored):,} Players ===')
print(scored[['PlayerID','Player_Persona','Churn_Probability_%','Risk_Tier']].head(10).to_string(index=False))
print(f'\n=== Risk Tier Distribution ===')
print(scored['Risk_Tier'].value_counts().sort_index().to_string())

In [ ]:
# Churn probability distribution chart
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Histogram
axes[0].hist(scored['Churn_Probability_%'], bins=50, color='#3498db', edgecolor='black', alpha=0.8)
axes[0].axvline(75, color='#e74c3c', linewidth=2, linestyle='--', label='CRM Threshold (75%)')
axes[0].set_xlabel('Churn Probability (%)')
axes[0].set_ylabel('Number of Players')
axes[0].set_title('Churn Probability Score Distribution', fontweight='bold')
axes[0].legend(fontsize=11)

# Risk tier pie
tier_counts = scored['Risk_Tier'].value_counts().sort_index()
tier_colors = ['#2ecc71', '#f39c12', '#e67e22', '#e74c3c']
axes[1].pie(tier_counts, labels=tier_counts.index, autopct='%1.1f%%',
            colors=tier_colors, startangle=90, shadow=True)
axes[1].set_title('Player Risk Tier Distribution', fontweight='bold')

plt.suptitle('Daily Churn Scoring Pipeline Output', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../data/processed/chart_churn_scoring_output.png', bbox_inches='tight')
plt.show()

---
## 7. 📤 CRM Export: Players with >75% Churn Probability
This is the **actionable output** that marketing feeds into their CRM for automated win-back emails/gifts.

In [ ]:
crm_threshold = 75.0
crm_list = scored[scored['Churn_Probability_%'] >= crm_threshold].copy()
crm_list = crm_list.sort_values('Churn_Probability_%', ascending=False)

# Add recommended action based on persona
action_map = {
    'At-Risk Newbies': 'Send tutorial completion bonus + 50% off first purchase email',
    'Weekend Warriors': 'Trigger weekend Double XP push notification',
    'Grinders': 'Award exclusive milestone badge + loot crate',
    'Whales': 'Assign VIP account manager; schedule personal outreach'
}
crm_list['Recommended_Action'] = crm_list['Player_Persona'].map(action_map)

# Export columns
export_cols = ['PlayerID', 'Player_Persona', 'Churn_Probability_%', 'Risk_Tier',
               'PlayTimeHours', 'SessionsPerWeek', 'InGamePurchases', 'Recommended_Action']
crm_export = crm_list[export_cols]

# Save
os.makedirs('../data/processed', exist_ok=True)
crm_export.to_csv('../data/processed/crm_churn_risk_players.csv', index=False)

print(f'=== CRM Export: {len(crm_export):,} Players with Churn Probability >= {crm_threshold}% ===')
print(crm_export.head(15).to_string(index=False))

print(f'\n=== Breakdown by Persona ===')
print(crm_export['Player_Persona'].value_counts().to_string())

print(f'\n\u2705 Exported to: data/processed/crm_churn_risk_players.csv')
print(f'\u2705 Ready to import into CRM (Salesforce, HubSpot, Braze, etc.)')

In [ ]:
# CRM list profile visualization
pcols = {'At-Risk Newbies':'#e74c3c','Weekend Warriors':'#f39c12','Grinders':'#3498db','Whales':'#9b59b6'}

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Persona split in CRM list
persona_counts = crm_export['Player_Persona'].value_counts()
pclrs = [pcols.get(p,'#95a5a6') for p in persona_counts.index]
axes[0].pie(persona_counts, labels=persona_counts.index, autopct='%1.1f%%',
            colors=pclrs, startangle=90, shadow=True)
axes[0].set_title(f'CRM List Composition\n({len(crm_export):,} players)', fontweight='bold')

# Churn prob distribution in CRM list
for persona in persona_counts.index:
    subset = crm_export[crm_export['Player_Persona']==persona]['Churn_Probability_%']
    sns.kdeplot(subset, ax=axes[1], label=persona, color=pcols.get(persona,'#95a5a6'), fill=True, alpha=0.3)
axes[1].set_xlabel('Churn Probability (%)')
axes[1].set_title('Churn Prob Distribution (CRM List)', fontweight='bold')
axes[1].legend(fontsize=8)

# Avg churn prob by persona
avg_prob = crm_export.groupby('Player_Persona')['Churn_Probability_%'].mean().sort_values()
bars = axes[2].barh(avg_prob.index, avg_prob.values,
                    color=[pcols.get(p,'#95a5a6') for p in avg_prob.index], edgecolor='black')
for b,v in zip(bars, avg_prob.values):
    axes[2].text(b.get_width()+0.3, b.get_y()+b.get_height()/2, f'{v:.1f}%', va='center', fontweight='bold')
axes[2].set_xlabel('Avg Churn Probability (%)')
axes[2].set_title('Avg Risk Score by Persona (CRM List)', fontweight='bold')

plt.suptitle('CRM Export Profile: High-Risk Players for Win-Back Campaigns', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../data/processed/chart_crm_export_profile.png', bbox_inches='tight')
plt.show()

---
## 8. Save Production Model

In [ ]:
os.makedirs('../models', exist_ok=True)
joblib.dump(best_model, f'../models/churn_scorer_{best.lower().replace(" ","_")}.pkl')
joblib.dump(le_dict, '../models/label_encoders.pkl')
joblib.dump(all_features, '../models/feature_columns.pkl')
print(f'\u2705 Production model saved: models/churn_scorer_{best.lower().replace(" ","_")}.pkl')
print(f'\u2705 Label encoders saved: models/label_encoders.pkl')
print(f'\u2705 Feature columns saved: models/feature_columns.pkl')

---
## 9. 📊 Pipeline Summary

### How Marketing Uses This Daily
```
1. Data team runs this pipeline every morning at 6 AM
2. Pipeline scores all active players (0-100%)
3. Players with score >= 75% are exported to CRM CSV
4. CRM auto-triggers persona-specific win-back campaigns:
   - At-Risk Newbies  → Tutorial bonus email
   - Weekend Warriors → Weekend XP push notification
   - Grinders         → Milestone reward in-game popup
   - Whales           → VIP manager personal outreach
5. Players are re-scored next day; interventions stop once score drops below 50%
```

In [ ]:
total_scored = len(scored)
crm_count = len(crm_export)
print('=' * 65)
print('DAILY CHURN SCORING PIPELINE SUMMARY')
print('=' * 65)
print(f'Total Players Scored:        {total_scored:,}')
print(f'Best Model:                  {best} (AUC={results[best]["auc"]:.4f})')
print(f'Features Used:               {len(all_features)} (incl. RFM + engineered)')
print(f'CRM Threshold:               >= {crm_threshold}%')
print(f'Players Flagged for CRM:     {crm_count:,} ({crm_count/total_scored*100:.1f}%)')
print(f'CRM Export File:             data/processed/crm_churn_risk_players.csv')
print(f'Production Model:            models/churn_scorer_{best.lower().replace(" ","_")}.pkl')
print('=' * 65)
print('\nPIPELINE READY FOR DAILY CRON SCHEDULING.')

In [ ]:
spark.stop()
print('SparkSession terminated.')